# 01 · Intro al Claude Agent SDK

**Objetivo:** instalar el SDK, autenticarse y ejecutar un `query()` mínimo. Verificar que recibimos `ResultMessage` con `stop_reason` real.

**Contexto del proyecto:** banca digital colombiana — flujo de cancelación de tarjeta. Persona primaria: Carlos, auditor de Customer Service. Marco regulatorio: Ley 1581/2012 (Habeas Data) + SFC Circular 029.

Documentación: https://code.claude.com/docs/en/agent-sdk/overview

## 1. Instalación

Desde la raíz de `notebooks/`:

```bash
python3 -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
export ANTHROPIC_API_KEY="sk-ant-..."
```

In [ ]:
# Verificamos versión instalada y que la API key está cargada.
import os, importlib.metadata as md

print('claude-agent-sdk', md.version('claude-agent-sdk'))
print('ANTHROPIC_API_KEY presente:', bool(os.environ.get('ANTHROPIC_API_KEY')))

## 2. Hello-world: un único `query()`

El SDK expone `query()` como async generator: cada `yield` es un mensaje del modelo (`AssistantMessage`, `ResultMessage`, etc.). El `ResultMessage` final trae `stop_reason` y `usage`.

In [ ]:
import asyncio
import nest_asyncio
from claude_agent_sdk import query, ClaudeAgentOptions, ResultMessage, AssistantMessage

nest_asyncio.apply()

async def hello():
    options = ClaudeAgentOptions(
        model='claude-opus-4-7',
        # No tools — el modelo solo debe responder texto.
        allowed_tools=[],
    )
    async for message in query(
        prompt='Responde en una frase: ¿qué es Habeas Data en Colombia?',
        options=options,
    ):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                # Bloques tipo TextBlock tienen .text
                text = getattr(block, 'text', None)
                if text:
                    print('[texto]', text)
        if isinstance(message, ResultMessage):
            print()
            print('stop_reason:', message.stop_reason)
            print('usage:', message.usage)
            print('cost USD:', round(message.total_cost_usd or 0, 6))

asyncio.run(hello())

## 3. Lo que importa para el prototipo

El SDK **ya emite señales deterministas verificables**: `stop_reason`, `usage.input_tokens`, `usage.output_tokens`. Esas son exactamente las señales sobre las que se construirá la UI determinista del proyecto.

En el siguiente notebook (`02_tools_banca.ipynb`) declaramos las herramientas reales del dominio: `account_status` y `cancel_product`.